In [1]:
# Cell 1 - Environment check and imports
import pandas as pd
import numpy as np
import tensorflow as tf
print(f"TensorFlow: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

df = pd.read_csv('/content/bq-results-20260323-203006-1774297820102.csv')
print(f"\nDataset shape: {df.shape}")
print(f"Columns: {df.shape[1]}")
print(f"Readmission rate: {df['readmitted_30d'].mean():.4f}")

TensorFlow: 2.20.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

Dataset shape: (406031, 53)
Columns: 53
Readmission rate: 0.1743


In [2]:
# Cell 2 - Data diagnosis
print(f"Total rows: {len(df)}")
print(f"Unique patients: {df['subject_id'].nunique()}")
print(f"Date range: {df['admittime'].min()} to {df['admittime'].max()}")
print(f"Readmission rate: {df['readmitted_30d'].mean():.4f}")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

Total rows: 406031
Unique patients: 180352
Date range: 2105-10-04 17:26:00 to 2214-12-15 19:11:00
Readmission rate: 0.1743

Missing values:
marital_status                10178
language                        555
insurance                      4602
admission_location                1
discharge_location            53135
num_lab_tests_24h             62672
num_abnormal_labs             62672
hemoglobin_min                91815
wbc_max                       93052
creatinine_max                99279
sodium_min                   103604
sodium_max                   103604
potassium_min                101293
potassium_max                101293
glucose_min                  107077
glucose_max                  107077
days_since_last_discharge    161291
days_to_next_admission       180352
dtype: int64


In [3]:
# Cell 3 - Preprocessing: handle missing values and prepare features
# Fill missing values
df['marital_status'] = df['marital_status'].fillna('UNKNOWN')
df['language'] = df['language'].fillna('UNKNOWN')
df['insurance'] = df['insurance'].fillna('UNKNOWN')
df['admission_location'] = df['admission_location'].fillna('UNKNOWN')
df['discharge_location'] = df['discharge_location'].fillna('UNKNOWN')

# Lab values - fill with median (clinical standard)
lab_cols = ['num_lab_tests_24h', 'num_abnormal_labs', 'hemoglobin_min',
            'wbc_max', 'creatinine_max', 'sodium_min', 'sodium_max',
            'potassium_min', 'potassium_max', 'glucose_min', 'glucose_max']
for col in lab_cols:
    df[col] = df[col].fillna(df[col].median())

# Historical features - fill with 0 (first-time patients)
hist_cols = ['days_since_last_discharge', 'num_admissions_last_30d',
             'num_admissions_last_90d', 'num_admissions_last_year',
             'total_prior_admissions', 'recent_admission_flag',
             'frequent_flyer_flag']
for col in hist_cols:
    df[col] = df[col].fillna(0)

# Fill remaining single missing values with median
df = df.fillna(df.median(numeric_only=True))

print(f"Missing values remaining: {df.isnull().sum().sum()}")
print(f"Shape: {df.shape}")

Missing values remaining: 0
Shape: (406031, 53)


In [4]:
# Cell 4 - Encode categorical features and define feature columns
from sklearn.preprocessing import LabelEncoder

categorical_cols = ['gender', 'race', 'marital_status', 'language', 'insurance',
                    'admission_location', 'discharge_location', 'admission_type']

for col in categorical_cols:
    if col in df.columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))

# Define feature columns (drop IDs, timestamps, targets)
drop_cols = ['subject_id', 'hadm_id', 'admittime', 'dischtime',
             'readmitted_30d', 'readmitted_60d', 'readmitted_90d',
             'days_to_next_admission']

feature_cols = [c for c in df.columns if c not in drop_cols]
print(f"Feature columns ({len(feature_cols)}):")
print(feature_cols)

Feature columns (45):
['gender', 'age', 'race', 'marital_status', 'language', 'insurance', 'admission_type', 'admission_location', 'discharge_location', 'los_hours', 'los_days', 'num_diagnoses', 'cci_mi', 'cci_chf', 'cci_pvd', 'cci_cvd', 'cci_dementia', 'cci_copd', 'cci_diabetes', 'cci_ckd', 'cci_cancer', 'num_lab_tests_24h', 'num_abnormal_labs', 'hemoglobin_min', 'wbc_max', 'creatinine_max', 'sodium_min', 'sodium_max', 'potassium_min', 'potassium_max', 'glucose_min', 'glucose_max', 'num_medications', 'polypharmacy_flag', 'anticoagulant_flag', 'insulin_flag', 'opioid_flag', 'antibiotic_flag', 'num_admissions_last_30d', 'num_admissions_last_90d', 'num_admissions_last_year', 'days_since_last_discharge', 'total_prior_admissions', 'recent_admission_flag', 'frequent_flyer_flag']


In [5]:
# Cell 5 - Drop race, finalize feature columns
feature_cols = [c for c in feature_cols if c != 'race']
print(f"Final feature columns ({len(feature_cols)}):")
print(feature_cols)

Final feature columns (44):
['gender', 'age', 'marital_status', 'language', 'insurance', 'admission_type', 'admission_location', 'discharge_location', 'los_hours', 'los_days', 'num_diagnoses', 'cci_mi', 'cci_chf', 'cci_pvd', 'cci_cvd', 'cci_dementia', 'cci_copd', 'cci_diabetes', 'cci_ckd', 'cci_cancer', 'num_lab_tests_24h', 'num_abnormal_labs', 'hemoglobin_min', 'wbc_max', 'creatinine_max', 'sodium_min', 'sodium_max', 'potassium_min', 'potassium_max', 'glucose_min', 'glucose_max', 'num_medications', 'polypharmacy_flag', 'anticoagulant_flag', 'insulin_flag', 'opioid_flag', 'antibiotic_flag', 'num_admissions_last_30d', 'num_admissions_last_90d', 'num_admissions_last_year', 'days_since_last_discharge', 'total_prior_admissions', 'recent_admission_flag', 'frequent_flyer_flag']


In [6]:
# Cell 6 - Build patient sequences for LSTM (fixed - per admission labels)
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import StandardScaler

MAX_SEQ_LEN = 10

# Sort by patient and admission time
df['admittime'] = pd.to_datetime(df['admittime'])
df = df.sort_values(['subject_id', 'admittime']).reset_index(drop=True)

# Scale features
scaler = StandardScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])

# Build sequences per patient
# For each admission, use all PREVIOUS admissions as context
# Label = that specific admission's readmitted_30d
X_sequences = []
y_labels = []
subject_ids = []

for subject_id, group in df.groupby('subject_id'):
    group = group.reset_index(drop=True)
    features = group[feature_cols].values
    labels = group['readmitted_30d'].values

    # For each admission in this patient's history
    for i in range(len(group)):
        # Sequence = admissions up to and including current
        seq = features[:i+1]
        label = labels[i]
        X_sequences.append(seq)
        y_labels.append(label)
        subject_ids.append(subject_id)

# Pad sequences
X_padded = pad_sequences(X_sequences, maxlen=MAX_SEQ_LEN,
                         dtype='float32', padding='pre', truncating='pre')
y_array = np.array(y_labels)

print(f"Total sequences: {len(X_padded)}")
print(f"Sequence shape: {X_padded.shape}")
print(f"Readmission rate: {y_array.mean():.4f}")

Total sequences: 406031
Sequence shape: (406031, 10, 44)
Readmission rate: 0.1743


In [7]:
print(df['admittime'].min())
print(df['admittime'].max())
print(df['admittime'].median())

2105-10-04 17:26:00
2214-12-15 19:11:00
2154-12-11 19:36:00


In [8]:
patient_first_admit = df.groupby('subject_id')['admittime'].min().reset_index()
patient_first_admit = patient_first_admit.rename(columns={'admittime': 'first_admittime'})

# Find percentile-based cutoff dates
sorted_dates = patient_first_admit['first_admittime'].sort_values()
train_cutoff = sorted_dates.quantile(0.70)
val_cutoff = sorted_dates.quantile(0.85)
print(f"Train cutoff (70%): {train_cutoff}")
print(f"Val cutoff (85%):   {val_cutoff}")

Train cutoff (70%): 2168-04-27 15:16:42
Val cutoff (85%):   2180-05-14 04:54:00


In [9]:
# Cell 7 - Chronological train/val/test split (admission level)
# Get admittime for each sequence (use original df order)
df_sorted = df.sort_values(['subject_id', 'admittime']).reset_index(drop=True)
admittimes = df_sorted['admittime'].values

train_cutoff = pd.Timestamp('2168-05-07')
val_cutoff = pd.Timestamp('2180-05-13')

train_idx = admittimes < train_cutoff
val_idx = (admittimes >= train_cutoff) & (admittimes < val_cutoff)
test_idx = admittimes >= val_cutoff

X_train, y_train = X_padded[train_idx], y_array[train_idx]
X_val, y_val = X_padded[val_idx], y_array[val_idx]
X_test, y_test = X_padded[test_idx], y_array[test_idx]

print(f"Train: {X_train.shape} | Readmission rate: {y_train.mean():.4f}")
print(f"Val:   {X_val.shape} | Readmission rate: {y_val.mean():.4f}")
print(f"Test:  {X_test.shape} | Readmission rate: {y_test.mean():.4f}")

Train: (270330, 10, 44) | Readmission rate: 0.1736
Val:   (60835, 10, 44) | Readmission rate: 0.1720
Test:  (74866, 10, 44) | Readmission rate: 0.1790


In [10]:
# Cell 8 - Build Bidirectional LSTM with Bahdanau Attention
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Bidirectional, LSTM,
                                      Dense, Dropout, Layer)
from tensorflow.keras.optimizers import Adam
import tensorflow.keras.backend as K

# Bahdanau Attention Layer
class BahdanauAttention(Layer):
    def __init__(self, units=64, **kwargs):
        super(BahdanauAttention, self).__init__(**kwargs)
        self.W = Dense(units)
        self.V = Dense(1)

    def call(self, encoder_output):
        # Score each time step
        score = self.V(tf.nn.tanh(self.W(encoder_output)))
        # Convert scores to weights (must sum to 1)
        attention_weights = tf.nn.softmax(score, axis=1)
        # Weighted sum of all time steps
        context_vector = attention_weights * encoder_output
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector, attention_weights

# Build model
def build_lstm_attention_model(seq_len, n_features):
    inputs = Input(shape=(seq_len, n_features))

    # Bidirectional LSTM
    lstm_out = Bidirectional(LSTM(64, return_sequences=True,
                                   dropout=0.3))(inputs)

    # Attention
    context_vector, attention_weights = BahdanauAttention(64)(lstm_out)

    # Dense layers
    x = Dense(32, activation='relu')(context_vector)
    x = Dropout(0.3)(x)
    output = Dense(1, activation='sigmoid')(x)

    model = Model(inputs=inputs, outputs=output)
    return model

model = build_lstm_attention_model(MAX_SEQ_LEN, len(feature_cols))
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 10, 44)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 10, 128)        │        55,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bahdanau_attention              │ [(None, 128), (None,   │         8,321 │
│ (BahdanauAttention)             │ 10, 1)]                │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 68,290 (266.76 KB)

 Trainable params: 68,290 (266.76 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
# Cell 9 - Compile and train the model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import numpy as np

# Handle class imbalance - give more weight to positive cases
neg = np.sum(y_train == 0)
pos = np.sum(y_train == 1)
class_weight = {0: 1.0, 1: neg/pos}
print(f"Class weight for positive class: {neg/pos:.2f}")

# Compile
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['AUC']
)

# Callbacks
early_stop = EarlyStopping(
    monitor='val_AUC',
    patience=5,
    mode='max',
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_AUC',
    factor=0.5,
    patience=3,
    mode='max',
    verbose=1
)

# Train
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=256,
    class_weight=class_weight,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

Class weight for positive class: 4.76
Epoch 1/30
1056/1056 ━━━━━━━━━━━━━━━━━━━━ 20s 12ms/step - AUC: 0.6756 - loss: 1.0636 - val_AUC: 0.6899 - val_loss: 0.6338 - learning_rate: 0.0010
Epoch 2/30
1056/1056 ━━━━━━━━━━━━━━━━━━━━ 12s 11ms/step - AUC: 0.6895 - loss: 1.0500 - val_AUC: 0.6938 - val_loss: 0.6183 - learning_rate: 0.0010
Epoch 3/30
1056/1056 ━━━━━━━━━━━━━━━━━━━━ 12s 11ms/step - AUC: 0.6940 - loss: 1.0455 - val_AUC: 0.6977 - val_loss: 0.6222 - learning_rate: 0.0010
Epoch 4/30
1056/1056 ━━━━━━━━━━━━━━━━━━━━ 12s 12ms/step - AUC: 0.6961 - loss: 1.0434 - val_AUC: 0.6992 - val_loss: 0.6020 - learning_rate: 0.0010
Epoch 5/30
1056/1056 ━━━━━━━━━━━━━━━━━━━━ 12s 12ms/step - AUC: 0.6978 - loss: 1.0415 - val_AUC: 0.6997 - val_loss: 0.6277 - learning_rate: 0.0010
Epoch 6/30
1056/1056 ━━━━━━━━━━━━━━━━━━━━ 12s 11ms/step - AUC: 0.7004 - loss: 1.0384 - val_AUC: 0.7006 - val_loss: 0.6062 - learning_rate: 0.0010
Epoch 7/30
1056/1056 ━━━━━━━━━━━━━━━━━━━━ 12s 11ms/step - AUC: 0.7015 - loss: 1.0373 -

In [12]:
# Cell 10 - Evaluate on test set
from sklearn.metrics import roc_auc_score

y_pred_proba = model.predict(X_test, verbose=0).flatten()
test_auroc = roc_auc_score(y_test, y_pred_proba)

print(f"Test AUROC: {test_auroc:.4f}")
print(f"\n--- Model Comparison ---")
print(f"Logistic Regression:       0.7021")
print(f"LR + Class Weights:        0.7025")
print(f"LR + SMOTE:                0.6984")
print(f"XGBoost (tuned):           0.7197")
print(f"LSTM + Attention:          {test_auroc:.4f}")
print(f"Target:                    0.7500")

Test AUROC: 0.7006

--- Model Comparison ---
Logistic Regression:       0.7021
LR + Class Weights:        0.7025
LR + SMOTE:                0.6984
XGBoost (tuned):           0.7197
LSTM + Attention:          0.7006
Target:                    0.7500


In [13]:
# Cell 11 - Tunable model builder
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

def build_lstm_attention_model_tunable(seq_len, n_features, lstm_units, dropout):
    inputs = Input(shape=(seq_len, n_features))
    lstm_out = Bidirectional(LSTM(lstm_units, return_sequences=True,
                                   dropout=dropout))(inputs)
    context_vector, attention_weights = BahdanauAttention(lstm_units)(lstm_out)
    x = Dense(64, activation='relu')(context_vector)
    x = Dropout(dropout)(x)
    output = Dense(1, activation='sigmoid')(x)
    model = Model(inputs=inputs, outputs=output)
    return model

print("Tunable model builder ready.")

Tunable model builder ready.


In [ ]:
# Cell 12 - Hyperparameter tuning loop
import itertools
from sklearn.metrics import roc_auc_score

lstm_units_options = [64, 128, 256]
dropout_options = [0.2, 0.3]
batch_size_options = [128, 256, 512]
lr_options = [0.001, 0.0005]

results = []

for lstm_units, dropout, batch_size, lr in itertools.product(
    lstm_units_options, dropout_options, batch_size_options, lr_options):

    print(f"Trying: units={lstm_units}, dropout={dropout}, batch={batch_size}, lr={lr}")

    tuned_model = build_lstm_attention_model_tunable(
        MAX_SEQ_LEN, len(feature_cols), lstm_units, dropout)

    neg = np.sum(y_train == 0)
    pos = np.sum(y_train == 1)
    class_weight = {0: 1.0, 1: neg/pos}

    tuned_model.compile(
        optimizer=Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['AUC']
    )

    callbacks = [
        EarlyStopping(monitor='val_AUC', patience=5, mode='max',
                      restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor='val_AUC', factor=0.5,
                          patience=3, mode='max', verbose=0)
    ]

    tuned_model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=30,
        batch_size=batch_size,
        class_weight=class_weight,
        callbacks=callbacks,
        verbose=0
    )

    y_pred = tuned_model.predict(X_test, verbose=0).flatten()
    auroc = roc_auc_score(y_test, y_pred)

    print(f"  AUROC: {auroc:.4f}")
    results.append({
        'lstm_units': lstm_units,
        'dropout': dropout,
        'batch_size': batch_size,
        'lr': lr,
        'auroc': auroc
    })

best = max(results, key=lambda x: x['auroc'])
print(f"\n--- Best Configuration ---")
print(f"LSTM units:  {best['lstm_units']}")
print(f"Dropout:     {best['dropout']}")
print(f"Batch size:  {best['batch_size']}")
print(f"LR:          {best['lr']}")
print(f"AUROC:       {best['auroc']:.4f}")

Trying: units=64, dropout=0.2, batch=128, lr=0.001
  AUROC: 0.7033
Trying: units=64, dropout=0.2, batch=128, lr=0.0005
  AUROC: 0.7018
Trying: units=64, dropout=0.2, batch=256, lr=0.001
  AUROC: 0.7017
Trying: units=64, dropout=0.2, batch=256, lr=0.0005
  AUROC: 0.7003
Trying: units=64, dropout=0.2, batch=512, lr=0.001
  AUROC: 0.7035
Trying: units=64, dropout=0.2, batch=512, lr=0.0005
  AUROC: 0.7022
Trying: units=64, dropout=0.3, batch=128, lr=0.001
  AUROC: 0.7026
Trying: units=64, dropout=0.3, batch=128, lr=0.0005
  AUROC: 0.7034
Trying: units=64, dropout=0.3, batch=256, lr=0.001
  AUROC: 0.7032
Trying: units=64, dropout=0.3, batch=256, lr=0.0005
  AUROC: 0.7037
Trying: units=64, dropout=0.3, batch=512, lr=0.001
  AUROC: 0.7034
Trying: units=64, dropout=0.3, batch=512, lr=0.0005
  AUROC: 0.7028
Trying: units=128, dropout=0.2, batch=128, lr=0.001
  AUROC: 0.7023
Trying: units=128, dropout=0.2, batch=128, lr=0.0005
  AUROC: 0.7036
Trying: units=128, dropout=0.2, batch=256, lr=0.001
 


KeyboardInterrupt



In [14]:
# Cell 14 - Best config with race, retrain and save
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import roc_auc_score

best_model_race = build_lstm_attention_model_tunable(
    MAX_SEQ_LEN, len(feature_cols), 128, 0.3)

neg = np.sum(y_train == 0)
pos = np.sum(y_train == 1)
class_weight = {0: 1.0, 1: neg/pos}

best_model_race.compile(
    optimizer=Adam(learning_rate=0.0005),
    loss='binary_crossentropy',
    metrics=['AUC']
)

callbacks = [
    EarlyStopping(monitor='val_AUC', patience=5, mode='max',
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_AUC', factor=0.5,
                      patience=3, mode='max', verbose=1)
]

best_model_race.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=128,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1
)

y_pred = best_model_race.predict(X_test, verbose=0).flatten()
auroc = roc_auc_score(y_test, y_pred)
print(f"\nFinal LSTM with race AUROC: {auroc:.4f}")

best_model_race.save('/content/lstm_with_race.keras')
#best_model_race.save('/content/lstm_with_race.h5')
print("Saved: lstm_with_race.keras")

Epoch 1/30
2112/2112 ━━━━━━━━━━━━━━━━━━━━ 29s 12ms/step - AUC: 0.6778 - loss: 1.0608 - val_AUC: 0.6896 - val_loss: 0.6347 - learning_rate: 5.0000e-04
Epoch 2/30
2112/2112 ━━━━━━━━━━━━━━━━━━━━ 25s 12ms/step - AUC: 0.6884 - loss: 1.0502 - val_AUC: 0.6934 - val_loss: 0.6239 - learning_rate: 5.0000e-04
Epoch 3/30
2112/2112 ━━━━━━━━━━━━━━━━━━━━ 24s 12ms/step - AUC: 0.6944 - loss: 1.0444 - val_AUC: 0.6978 - val_loss: 0.6259 - learning_rate: 5.0000e-04
Epoch 4/30
2112/2112 ━━━━━━━━━━━━━━━━━━━━ 24s 11ms/step - AUC: 0.6973 - loss: 1.0412 - val_AUC: 0.6986 - val_loss: 0.6205 - learning_rate: 5.0000e-04
Epoch 5/30
2112/2112 ━━━━━━━━━━━━━━━━━━━━ 25s 12ms/step - AUC: 0.7002 - loss: 1.0385 - val_AUC: 0.7014 - val_loss: 0.5997 - learning_rate: 5.0000e-04
Epoch 6/30
2112/2112 ━━━━━━━━━━━━━━━━━━━━ 40s 11ms/step - AUC: 0.7018 - loss: 1.0365 - val_AUC: 0.7021 - val_loss: 0.6185 - learning_rate: 5.0000e-04
Epoch 7/30
2112/2112 ━━━━━━━━━━━━━━━━━━━━ 25s 12ms/step - AUC: 0.7037 - loss: 1.0346 - val_AUC: 0.70